In [1]:
print("""
@File         : grouping_and_calculating_multiple_columns.ipynb
@Author(s)    : Stephen CUI
@LastEditor(s): Stephen CUI
@CreatedTime  : 2025-01-04 21:05:58
@Email        : cuixuanstephen@gmail.com
@Description  : 分组并计算多列
""")


@File         : grouping_and_calculating_multiple_columns.ipynb
@Author(s)    : Stephen CUI
@LastEditor(s): Stephen CUI
@CreatedTime  : 2025-01-04 21:05:58
@Email        : cuixuanstephen@gmail.com
@Description  : 分组并计算多列



In [2]:
import pandas as pd

In [3]:
df = pd.DataFrame([
    ["North", "Widget A", "Jan", 10, 2],
    ["North", "Widget B", "Jan", 4, 0],
    ["South", "Widget A", "Jan", 8, 3],
    ["South", "Widget B", "Jan", 12, 8],
    ["North", "Widget A", "Feb", 3, 0],
    ["North", "Widget B", "Feb", 7, 0],
    ["South", "Widget A", "Feb", 11, 2],
    ["South", "Widget B", "Feb", 13, 4],
], columns=["region", "widget", "month", "sales", "returns"])
df = df.convert_dtypes(dtype_backend="numpy_nullable")

df

,region,widget,month,sales,returns
0,North,Widget A,Jan,10,2
1,North,Widget B,Jan,4,0
2,South,Widget A,Jan,8,3
3,South,Widget B,Jan,12,8
4,North,Widget A,Feb,3,0
5,North,Widget B,Feb,7,0
6,South,Widget A,Feb,11,2
7,South,Widget B,Feb,13,4


In [4]:
df.groupby('widget').sum()

,region,month,sales,returns
widget,,,,
Widget A,NorthSouthNorthSouth,JanJanFebFeb,32,7
Widget B,NorthSouthNorthSouth,JanJanFebFeb,36,12


虽然销售额和回报率看起来不错，但地区和月份列最终也被求和，使用与 Python 处理字符串时相同的求和逻辑。

避免此问题的一种方法是在 `df.groupby("widget")` 调用之后选择要聚合的列，以更明确地说明这些列：

In [5]:
df.groupby('widget')[['sales', 'returns']].agg('sum')

,sales,returns
widget,,
Widget A,32,7
Widget B,36,12


或者，你可以使用 `pd.NamedAgg` 类。虽然更冗长，但使用 `pd.NamedAgg` 可以重命名想要在输出中看到的列（例如，可能希望看到 `sales_total`，而不是 `sales`）：

In [6]:
df.groupby('widget').agg(
    sales_total=pd.NamedAgg(column='sales', aggfunc='sum'),
    return_totals=pd.NamedAgg(column='returns', aggfunc='sum')
)

,sales_total,return_totals
widget,,
Widget A,32,7
Widget B,36,12


`pd.core.groupby.DataFrameGroupBy`  的另一个值得一提的功能是它能够处理多个组参数。

In [7]:
df.groupby(['widget', 'region']).agg(
    sales_total=pd.NamedAgg(column='sales', aggfunc='sum'),
    return_totals=pd.NamedAgg(column='returns', aggfunc='sum')
)

sales_total  return_totals
widget   region                            
Widget A North            13              2
         South            19              5
Widget B North            11              0
         South            25             12

使用 `pd.core.groupby.DataFrameGroupBy.agg`，可以应用的函数数量没有限制。

In [10]:
import numpy as np

df.groupby(["widget", "region"]).agg(
    sales_total=pd.NamedAgg("sales", "sum"),
    returns_total=pd.NamedAgg("returns", 'sum'),
    sales_min=pd.NamedAgg("sales", "min"),
    returns_min=pd.NamedAgg("returns", "min"),
)

sales_total  returns_total  sales_min  returns_min
widget   region                                                    
Widget A North            13              2          3            0
         South            19              5          8            2
Widget B North            11              0          4            0
         South            25             12         12            4

虽然内置的缩减函数和转换函数与 group by 配合使用非常有用，但有时仍需要使用自己的自定义函数。当发现某个算法足以满足您在本地分析中尝试的需要，但很难推广到所有用例时，这会特别有用。

pandas 中一个常用的函数是 `mode`，虽然有 `pd.Series.mode`方法，但 group by 并没有提供该函数。使用 `pd.Series.mode`，返回的类型始终是 pd.Series，无论是否只有一个最常出现的值：

In [12]:
pd.Series([0, 1, 1]).mode()

0    1
dtype: int64

既然有 `pd.Series.mode`，为什么 pandas 在执行 group by 时不提供类似的功能？从 pandas 开发人员的角度来看，原因很简单；没有单一的方法来解释 group by 应该返回什么。

In [14]:
df = pd.DataFrame([
    ["group_a", 42],
    ["group_a", 555],
    ["group_a", 42],
    ["group_a", 555],
    ["group_b", 0],
], columns=["group", "value"])

df

,group,value
0,group_a,42
1,group_a,555
2,group_a,42
3,group_a,555
4,group_b,0


我们需要回答的问题是 group_a 的模式应该返回什么？一种可能的解决方案是返回一个包含 42 和 555 的列表（或任何 Python 序列）。这种方法的缺点是返回的 dtype 将是 object，这种数据类型是有缺点的。

第二个期望是让 Pandas 只选择其中一个值。当然，这引出了一个问题：Pandas 应该如何做出决定选择值 42 还是 555 更合适。

第三个期望是返回在聚合后标签 `group_a` 在结果行索引中出现两次的内容。

pandas 不会试图解决所有这些期望并将其编入 API 的一部分，而是让你完全自行决定如何实现模式函数，只要遵守聚合减少到每个组一个值的期望即可。

In [16]:
def scalar_or_list_mode(ser: pd.Series):
    result = ser.mode()
    if len(result) > 1:
        return result.tolist()
    elif len(result) == 1:
        return result.iloc[0]
    
    return pd.NA


def scalar_or_bust_mode(ser: pd.Series):
    result = ser.mode()
    if len(result) == 0:
        return pd.NA
    
    return result.iloc[0]

In [18]:
df.groupby('group').agg(
    scalar_or_list=pd.NamedAgg('value', scalar_or_list_mode),
    scalar_or_bust=pd.NamedAgg('value', scalar_or_bust_mode)
)

,scalar_or_list,scalar_or_bust
group,,
group_a,"[42, 555]",42
group_b,0,0
